# Customer Churn Prediction — Data Analysis & Machine Learning

**Project goal:** explore customer churn patterns, prepare the data, train a classification model, and evaluate its predictions.

**Dataset:** `data/Customer Churn.csv`  
**Target column:** `Churn` (`Yes` means the customer churned; `No` means they stayed).

> Run the notebook from the project folder. If your CSV is stored elsewhere, update `DATA_PATH` in the next cell.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)
import joblib

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 2. Load the dataset

In [ ]:
# Expected location when this notebook is inside the project folder:
DATA_PATH = Path("data") / "Customer Churn.csv"

# If running from a different working directory, try the common alternative.
if not DATA_PATH.exists():
    alternative = Path("Customer Churn.csv")
    if alternative.exists():
        DATA_PATH = alternative

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the dataset at {DATA_PATH.resolve()}. "
        "Place 'Customer Churn.csv' in the project's data folder or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())

## 3. Understand and inspect the data

In [ ]:
display(df.info())
display(df.describe(include="all").T)
print("Duplicate rows:", df.duplicated().sum())
print("\\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

## 4. Clean the data

In [ ]:
df = df.copy()

# TotalCharges can be stored as text and may contain blank strings.
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove duplicate records, if any.
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicate rows:", df.shape)
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

## 5. Explore churn distribution

In [ ]:
if "Churn" not in df.columns:
    raise KeyError("Expected target column 'Churn' was not found in the dataset.")

churn_counts = df["Churn"].value_counts(dropna=False)
display(churn_counts.to_frame("customers"))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", order=df["Churn"].value_counts().index)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

print("Churn rate (%):", round((df["Churn"].astype(str).str.lower() == "yes").mean() * 100, 2))

## 6. Explore numeric customer attributes

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)
    df[numeric_cols].hist(figsize=(12, 8), bins=25, edgecolor="black")
    plt.suptitle("Distributions of Numeric Features")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns were found.")

## 7. Churn patterns by customer features

In [ ]:
candidate_features = [
    col for col in ["Contract", "InternetService", "PaymentMethod",
                    "SeniorCitizen", "Partner", "Dependents", "TechSupport"]
    if col in df.columns
]

for col in candidate_features:
    summary = pd.crosstab(df[col], df["Churn"], normalize="index") * 100
    if "Yes" in summary.columns:
        summary = summary.sort_values("Yes", ascending=False)
    display(summary.round(1).rename_axis(col))

    ax = pd.crosstab(df[col], df["Churn"], normalize="index").plot(
        kind="bar", stacked=True, figsize=(8, 4), ylim=(0, 1)
    )
    ax.set_title(f"Churn share by {col}")
    ax.set_ylabel("Share of customers")
    ax.legend(title="Churn", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## 8. Prepare features and target

In [ ]:
# Convert the target to a consistent binary value.
target = df["Churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
valid_target = target.notna()
model_df = df.loc[valid_target].copy()
y = target.loc[valid_target].astype(int)

# CustomerID is an identifier, not a useful predictive feature.
drop_cols = ["Churn"]
if "customerID" in model_df.columns:
    drop_cols.append("customerID")
X = model_df.drop(columns=drop_cols)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical feature count:", len(categorical_features))
print("Target counts (0=No churn, 1=Churn):")
display(y.value_counts().sort_index().rename(index={0: "No churn", 1: "Churn"}).to_frame("customers"))

## 9. Train-test split and preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 10. Train the Logistic Regression model

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

model.fit(X_train, y_train)
print("Model training completed.")

## 11. Evaluate model performance

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}
metrics_df = pd.DataFrame({"Metric": metrics.keys(), "Score": metrics.values()})
display(metrics_df.style.format({"Score": "{:.3f}"}))

print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted no churn", "Predicted churn"],
            yticklabels=["Actual no churn", "Actual churn"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.tight_layout()
plt.show()

## 12. Inspect feature influence (Logistic Regression coefficients)

In [ ]:
# Coefficients are shown after one-hot encoding, so category levels become separate features.
try:
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    coefficients = model.named_steps["classifier"].coef_[0]
    coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": coefficients})
    coef_df["AbsoluteCoefficient"] = coef_df["Coefficient"].abs()
    display(coef_df.sort_values("AbsoluteCoefficient", ascending=False).head(20))

    top = coef_df.sort_values("AbsoluteCoefficient", ascending=False).head(15)
    plt.figure(figsize=(9, 6))
    sns.barplot(data=top.sort_values("Coefficient"), x="Coefficient", y="Feature")
    plt.title("Largest Logistic Regression Coefficients")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Could not display coefficients:", exc)

## 13. Save the trained model

In [ ]:
# Save in the project's models folder.
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "churn_pipeline.joblib"

joblib.dump(model, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH.resolve()}")

## 14. Example prediction for one customer

In [ ]:
# Demonstration only: uses a real row from the test set.
sample_customer = X_test.iloc[[0]]
sample_probability = model.predict_proba(sample_customer)[0, 1]
sample_prediction = model.predict(sample_customer)[0]

if sample_probability < 0.30:
    risk_band = "Low"
elif sample_probability < 0.60:
    risk_band = "Medium"
else:
    risk_band = "High"

print("Predicted churn:", "Yes" if sample_prediction == 1 else "No")
print(f"Estimated churn probability: {sample_probability:.1%}")
print("Risk band:", risk_band)
display(sample_customer)

## 15. Conclusion

This notebook covers the full analytical workflow: data loading, cleaning, exploratory analysis, preprocessing, model training, evaluation, and saving the fitted pipeline.

**Important interpretation note:** churn predictions are estimates, not guarantees. Accuracy alone is not sufficient; precision, recall, F1-score, ROC-AUC, and business costs should also be considered. The best risk threshold depends on how the business balances missed churners against unnecessary retention offers.

### Suggested next steps
- Compare Logistic Regression with Random Forest and Gradient Boosting.
- Tune model hyperparameters using cross-validation.
- Review the precision-recall trade-off before setting a production threshold.
- Monitor model performance when new customer data becomes available.